In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd

PROMPT_TEMPLATE = """You are a data processing pipeline. Follow these steps in order:
Step 1: Check the 'Surface Albedo' column. If all values are 0.12, proceed to Step 2. If not, stop and report 'ALBEDO_VARIANT'.
Step 2: Find the row with the highest 'Solar Zenith Angle'.
Step 3: Check the 'Wind Speed' for that specific row. If it is above 3.0, output 'HIGH_WIND_PEAK'. If below, output 'CALM_PEAK'.

Only output the final status word from Step 3 or the error from Step 1.

Here is the data:
{data_snippet}
"""

@kbench.task(name="single_planning_step", store_task=False)
def single_planning_step(llm, data_snippet: str, expected_output: str) -> bool:
    """Tests a single instance of the planning task."""
    prompt = PROMPT_TEMPLATE.format(data_snippet=data_snippet)
    response = llm.prompt(prompt)
    return expected_output.lower() in response.lower()

@kbench.task(name="executive_function_test_multi")
def executive_function_test(llm) -> float:
    """Evaluates the LLM's ability to follow a multi-step plan with conditional logic."""
    test_cases = [
        # Case 1: Calm Peak - Max Zenith Angle (50) has Wind Speed (2.1) <= 3.0
        {"data_snippet": "Surface Albedo,Solar Zenith Angle,Wind Speed\n0.12,30,4.0\n0.12,50,2.1\n0.12,45,3.5", "expected_output": "CALM_PEAK"},
        # Case 2: High Wind Peak - Max Zenith Angle (60) has Wind Speed (3.8) > 3.0
        {"data_snippet": "Surface Albedo,Solar Zenith Angle,Wind Speed\n0.12,60,3.8\n0.12,50,2.1\n0.12,45,3.5", "expected_output": "HIGH_WIND_PEAK"},
        # Case 3: Albedo Variant - Not all albedo values are 0.12
        {"data_snippet": "Surface Albedo,Solar Zenith Angle,Wind Speed\n0.12,30,4.0\n0.11,50,2.1\n0.12,45,3.5", "expected_output": "ALBEDO_VARIANT"},
        # Case 4: Calm Peak - Max Zenith Angle (80) has Wind Speed (1.0) <= 3.0
        {"data_snippet": "Surface Albedo,Solar Zenith Angle,Wind Speed\n0.12,80,1.0\n0.12,75,5.0", "expected_output": "CALM_PEAK"},
        # Case 5: High Wind Peak - Max Zenith Angle (90) has Wind Speed (9.9) > 3.0
        {"data_snippet": "Surface Albedo,Solar Zenith Angle,Wind Speed\n0.12,20,1.0\n0.12,90,9.9", "expected_output": "HIGH_WIND_PEAK"},
        # Case 6: Albedo Variant - Not all albedo values are 0.12
        {"data_snippet": "Surface Albedo,Solar Zenith Angle,Wind Speed\n0.12,20,1.0\n0.12,90,9.9\n0.13,50,3.0", "expected_output": "ALBEDO_VARIANT"},
        # Case 7: Calm Peak (Edge Case) - Max Zenith Angle (44) has Wind Speed (3.0) <= 3.0
        {"data_snippet": "Surface Albedo,Solar Zenith Angle,Wind Speed\n0.12,44,3.0\n0.12,42,3.1\n0.12,43,2.9", "expected_output": "CALM_PEAK"},
        # Case 8: High Wind Peak (Edge Case) - Max Zenith Angle (55) has Wind Speed (3.01) > 3.0
        {"data_snippet": "Surface Albedo,Solar Zenith Angle,Wind Speed\n0.12,50,2.9\n0.12,55,3.01\n0.12,52,3.0", "expected_output": "HIGH_WIND_PEAK"},
        # Case 9: Albedo Variant - Not all albedo values are 0.12
        {"data_snippet": "Surface Albedo,Solar Zenith Angle,Wind Speed\n0.10,50,2.9\n0.12,55,3.01", "expected_output": "ALBEDO_VARIANT"},
        # Case 10: Calm Peak (Single Row) - Max Zenith Angle (90) has Wind Speed (2.5) <= 3.0
        {"data_snippet": "Surface Albedo,Solar Zenith Angle,Wind Speed\n0.12,90,2.5", "expected_output": "CALM_PEAK"},
    ]
    df = pd.DataFrame(test_cases)

    with kbench.client.enable_cache():
        runs = single_planning_step.evaluate(
            llm=[llm],
            evaluation_data=df,
            n_jobs=2,
            remove_run_files=True
        )

    eval_df = runs.as_dataframe()
    if eval_df.empty or 'result' not in eval_df.columns:
        return 0.0
        
    accuracy = float(eval_df.result.mean())
    return accuracy

executive_function_test.run(kbench.llm)